## _Scrapper_ | Site Anvisa de Registro e Notificações de Medicamentos
### Definição de Variáveis

In [1]:
from pathlib import Path

DATASET_DIR = 'samples/dataset'
DATASET_FILENAME = 'Anvisa_REGISTRO_MEDICAMENTOS.csv'

ROOT_DIR = Path().resolve().__str__()
DATASET_PATH = ROOT_DIR + '/' + DATASET_DIR
Path.mkdir(Path(DATASET_PATH), parents=True, exist_ok=True)
DATASET_FILEPATH = Path(DATASET_PATH + '/' + DATASET_FILENAME)
TOTAL_ELEMENTS = 50000

### Definição da request

In [12]:
url = "https://consultas.anvisa.gov.br/api/consulta/medicamento/produtos/"

In [13]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:152.0) Gecko/20100101 Firefox/152.0",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br, zstd",
    "If-Modified-Since": "Mon, 26 Jul 1997 05:00:00 GMT",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Authorization": "Guest",
    "x-dtpc": "4$266473457_314h85vLUAVIHKFRGMWAOEQDLOVUGPFNFINOJVM-0e0",
    "x-dtreferer": "https://consultas.anvisa.gov.br/",
    "Connection": "keep-alive",
    "Referer": "https://consultas.anvisa.gov.br/",
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin",
    "Priority": "u=0",
}

In [14]:
params = [
    {
        "column": "",
        "count": TOTAL_ELEMENTS,
        "filter[checkNotificado]": "true", # Notificado
        "filter[checkRegistrado]": "true",
        "filter[situacaoRegistro]": "V", # Ativo
        "order": "asc",
        "page": 1,
    },
    {
        "column": "",
         "count": TOTAL_ELEMENTS,
         "filter[checkNotificado]": "true", # Notificado
         "filter[checkRegistrado]": "true",
         "filter[situacaoRegistro]": "C", # Inativo
         "order": "asc",
         "page": 1,
     },
]

### Execução da Request

In [3]:
import cloudscraper as cs

try:
    scrapper = cs.create_scraper()
    responses = [scrapper.get(url, params=param, headers=headers) for param in params]

    print(f'Raw data retrieved from {url}')

except cs.exceptions.CaptchaException as e:
    print(e)

Raw data retrieved from https://consultas.anvisa.gov.br/api/consulta/medicamento/produtos/


### Tratamento dos dados

In [15]:
medicacao = []
dataVencimentoRegistro = []
situacaoApresentacao = []
categoriaRegulatoriaDescricao = []
categoriaRegulatoriaCategoria = []
tipoAutorizacao = []
razaoSocial = []
cnpjFormatado = []
numeroProcessoFormatado = []

In [4]:
import json

for response_json in responses:
    response_json = json.loads(response_json.text)
    for item in response_json['content']:
        produto = dict(item['produto'])
        medicacao.append(produto['nome'])
        situacaoApresentacao.append(produto['situacaoApresentacao'])
        dataVencimentoRegistro.append(str(produto['dataVencimentoRegistro']).split('T')[0])
        categoriaRegulatoriaDescricao.append(produto['categoriaRegulatoria']['descricao'])
        categoriaRegulatoriaCategoria.append(produto['categoriaMedicamentoNotificado'])
        tipoAutorizacao.append(produto['tipoAutorizacao'])

        empresa = dict(item['empresa'])
        razaoSocial.append(empresa['razaoSocial'])
        cnpjFormatado.append(empresa['cnpjFormatado'])

        processo = dict(item['processo'])
        numeroProcessoFormatado.append(item['processo']['numeroProcessoFormatado'])


In [5]:
import pandas as pd

df = pd.DataFrame(data={'medicamento': medicacao,
                   'situacao': situacaoApresentacao,
                   'data_Vencimento_Registro': dataVencimentoRegistro,
                   'categoria_Regulatoria_Descricao': categoriaRegulatoriaDescricao,
                   'tipo_Autorizacao': tipoAutorizacao,
                   'razao_Social': razaoSocial,
                   'cnpj_Formatado': cnpjFormatado,
                   'numero_Processo_Formatado': numeroProcessoFormatado,
                   })

### Exibição das Informações

In [7]:
docs = [row[1].to_markdown(tablefmt=None) for row in df.iterrows()]

In [8]:
print('\n'.join(docs[:5]))

                                 0
-------------------------------  -----------------------------------
medicamento                      IROSÊ
situacao                         Ativo
data_Vencimento_Registro         2033-08-01
categoria_Regulatoria_Descricao  Similar
tipo_Autorizacao                 REGISTRADO
razao_Social                     ALTHAIA S.A. INDUSTRIA FARMACEUTICA
cnpj_Formatado                   48.344.725/0007-19
numero_Processo_Formatado        25351.305631/2023-68
                                 1
-------------------------------  --------------------------------
medicamento                      SOLIQUA
situacao                         Ativo
data_Vencimento_Registro         2027-07-01
categoria_Regulatoria_Descricao  Biológico
tipo_Autorizacao                 REGISTRADO
razao_Social                     SANOFI MEDLEY FARMACÊUTICA LTDA.
cnpj_Formatado                   10.588.595/0010-92
numero_Processo_Formatado        25351.411883/2019-49
                              

### Exportação para CSV

In [9]:
df.to_csv(DATASET_FILEPATH, index=False, encoding='utf-8')

### Checksum

In [10]:
import hashlib
print(f'{DATASET_FILEPATH.name}: {hashlib.md5(open(DATASET_FILEPATH, 'rb').read()).hexdigest()}')

Anvisa_REGISTRO_MEDICAMENTOS.csv: fa7c0dbc84662750bf5e1468aae9b6e9
